## Imports

In [39]:
from pathlib import Path
import fitz
import re
from collections import Counter
print("Done")

Done


### Detect project root

In [2]:
CURRENT_DIR = Path.cwd()

ROOT_DIR = next(
    (
        path
        for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "data").is_dir() and (path / "config.py").is_file()
    ),
    None
)

if ROOT_DIR is None:
    raise FileNotFoundError("Project root could not be detected.")

print("Project root:", ROOT_DIR)

Project root: c:\Users\DELL\OneDrive\Documents\paypal-copilot-1


### Path setup

In [4]:
DATA_DIR = ROOT_DIR / "data"

KNOWLEDGE_BASE_DIR = DATA_DIR / "knowledge_base"
PLAYBOOK_DIR = DATA_DIR / "playbooks"

RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

KB_PDF_PATH = KNOWLEDGE_BASE_DIR / "PayPal_Dispute_Knowledge_Base.pdf"

VECTOR_STORE_DIR = ROOT_DIR / "rag" / "vector_store"

print("Done")

Done


### Creating required dictionaries and validate paths

In [6]:
PROCESSED_DATA_DIR.mkdir(parents = True, exist_ok = True)
VECTOR_STORE_DIR.mkdir(parents = True , exist_ok = True)

assert KB_PDF_PATH.exists(), "Knowledge base PDF not found"

print("PDF",KB_PDF_PATH)
print("playbboks:",PLAYBOOK_DIR)
print("proceesd Data :",PROCESSED_DATA_DIR)
print("vector store :",VECTOR_STORE_DIR)

PDF c:\Users\DELL\OneDrive\Documents\paypal-copilot-1\data\knowledge_base\PayPal_Dispute_Knowledge_Base.pdf
playbboks: c:\Users\DELL\OneDrive\Documents\paypal-copilot-1\data\playbooks
proceesd Data : c:\Users\DELL\OneDrive\Documents\paypal-copilot-1\data\processed
vector store : c:\Users\DELL\OneDrive\Documents\paypal-copilot-1\rag\vector_store


### Loading the Knowledge Base

In [14]:
pdf_docs = []
with fitz.open(KB_PDF_PATH) as pdf:
    for page_number,page in enumerate(pdf, start = 1):
        page_text = page.get_text("text").strip()
        if not page_text:
            continue
        pdf_docs.append({
            "text":page_text,
            "metadata":{
                "sourse":KB_PDF_PATH.name,
                "page":page_number,
                "document_type":"knowledge_base"
            }
        })

print("Total extracted pages:",len(pdf_docs))

Total extracted pages: 9


### Inspecting the extracted text

In [20]:
def inspecting_data(documents):
    for idx,doc in enumerate(documents[:2]):
        print(f"idx :{idx} and Metadata {doc["metadata"]}\n")
        print(doc["text"][:1000])
        print("="*80)

inspecting_data(pdf_docs)


idx :0 and Metadata {'sourse': 'PayPal_Dispute_Knowledge_Base.pdf', 'page': 1, 'document_type': 'knowledge_base'}

PayPal Dispute Knowledge Base
for PayPal Copilot (RAG corpus)  |  accessed 4 Nov 2026
Original summaries of public PayPal documentation. Not official PayPal text. Policies vary by country and change over time.
1
PayPal Dispute Knowledge Base
Item not received, not as described, unauthorized claims, evidence, Seller Protection and merchant
actions. Compiled for retrieval by the PayPal Copilot dispute agent.
How to use this document. Every entry is self-contained and starts with a metadata line: an ID, a type
(overview, api, policy, guidance or glossary), the dispute reason it applies to, and its source. Entries of type
guidance are the project builder's practical heuristics, not PayPal rules. Text is paraphrased from public
PayPal pages (accessed 4 Nov 2026); the United States Seller Protection policy is used throughout. Always
defer to PayPal's current published terms.
Con

### Inspecting PDF extraction Quality

In [21]:
def inspecting_quality(documents):
    for doc in documents:
        text = doc["text"]
        print(
            f"Page: {doc["metadata"]["page"]} | "
            f"Chracters :{len(text)}"
        )

inspecting_quality(pdf_docs)

Page: 1 | Chracters :3552
Page: 2 | Chracters :3681
Page: 3 | Chracters :3277
Page: 4 | Chracters :3568
Page: 5 | Chracters :3507
Page: 6 | Chracters :4047
Page: 7 | Chracters :3331
Page: 8 | Chracters :3334
Page: 9 | Chracters :2537


### Inspecting policy RAG


In [27]:
def inspect_policy(documents):
    pdf_text = "\n".join(doc["text"]for doc in documents)
    policy_ids = sorted(set(re.findall(r"\b[A-Z]\d+\b",pdf_text)))
    print("Policy refrences found :",policy_ids)

inspect_policy(pdf_docs)

Policy refrences found : ['A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B2', 'B3', 'B4', 'B5', 'C1', 'C2', 'C3', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'F1', 'F2', 'F3', 'F4', 'G1']


### Loading the playbooks

In [29]:
playbook_files = {
    "item_not_received":PLAYBOOK_DIR/"item_not_received.md",
    "not_as_described":PLAYBOOK_DIR/"not_as_described.md",
    "unauthorized":PLAYBOOK_DIR/"unauthorized.md"
    }

print("Done")

Done


### Extracting playbbok content

In [41]:
def extreact_plybook(playbook):
    playbook_docs = []
    for dispute_type, file_path in playbook.items():
        if not file_path.exists():
            raise FileNotFoundError(f"Missing plabook :{file_path}")
        text = file_path.read_text(encoding = "utf-8").strip()
        playbook_docs.append({
            "text":text,
            "metadata":{
                "source":file_path.name,
                "document_type":"playbook",
                "dispute_type":dispute_type
            }
        })
    return playbook_docs

playbook_docs = extreact_plybook(playbook_files)
print("Play books :",len(playbook_docs))

Play books : 3


### Inspect playbook content

In [37]:
def inspect_playbooks(playbooks):
    for ply_doc in playbooks:
        print("Source:",ply_doc["metadata"]["source"])
        print("Dispute :",ply_doc["metadata"]["dispute_type"])
        print("Characters :",len(ply_doc["text"]))
        print(ply_doc["text"][:500])
        print("="*80)

inspect_playbooks(playbook_docs)

Source: item_not_received.md
Dispute : item_not_received
Characters : 916
# Playbook: Item not received (INR)

> Write this in your own words. It is original work and is committed to the repo. Keep every section short and concrete.

## When this applies
(What the buyer is claiming, and which dispute reason code it maps to.)

## Evidence that wins
(List the specific items: tracking, delivery status, signature, address match, dates. One line each.)

## Evidence that is weak or hurts
(What looks suspicious or unhelpful: address mismatch, in transit, no tracking, returned
Source: not_as_described.md
Dispute : not_as_described
Characters : 916
# Playbook: Not as described (SNAD)

> Write this in your own words. It is original work and is committed to the repo. Keep every section short and concrete.

## When this applies
(What the buyer is claiming, and which dispute reason code it maps to.)

## Evidence that wins
(List the specific items: tracking, delivery status, signature, address match,

### Combine the documents

In [38]:
def combine_docs(pdf_docs,playbook_docs):
    all_docs = pdf_docs + playbook_docs
    return all_docs

combined_docs = combine_docs(pdf_docs,playbook_docs)
print("Total documents",len(combined_docs))

Total documents 12
